# XTTS-v2'yi kendi sesine ince ayarlama (ücretsiz Colab)

**Önce:** Menüden *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seç.

**Google Drive'a hazırlık:** Drive'ında `xtts-ft` adlı bir klasör aç, içine kayıtlarını koy:
`kayit-tr.m4a` (Türkçe, birden fazlaysa `kayit-tr-1.m4a`, `kayit-tr-2.m4a`) ve `kayit-en.m4a` (İngilizce).
Dil dosya adındaki `-tr` / `-en` kısmından anlaşılır.

Sonra hücreleri yukarıdan aşağı sırayla çalıştır (her hücrenin solundaki ▶ düğmesi). Toplam ~40-60 dakika.
Sonuç: Drive'da `xtts-ft/xtts-arjin.zip`. Bunu Mac'e indir, kullanım en alttaki hücrede.

Lisans: XTTS-v2 ağırlıkları CPML (ticari olmayan kullanım).

## 1. Kurulum (5-8 dk). Bittikten sonra 'oturumu yeniden başlat' uyarısı çıkarsa yeniden başlatıp 2. hücreden devam et.

In [ ]:
!pip install -q coqui-tts openai-whisper six
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Drive'ı bağla ve ayarlar

In [ ]:
import os, glob, json, re, shutil
os.environ["COQUI_TOS_AGREED"] = "1"
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/xtts-ft"
WORK = "/content/work"
EPOCHS = 10          # az veri için 8-12 iyi; ses robotikleşirse azalt, benzemezse artır
BATCH = 3
LR = 5e-6
os.makedirs(WORK, exist_ok=True)
kayitlar = sorted(f for f in glob.glob(DRIVE + "/kayit-*") if re.search(r"-(tr|en)", os.path.basename(f)))
print("\n".join(kayitlar) or "Kayıt bulunamadı! Drive'da xtts-ft/kayit-tr.m4a var mı?")

## 3. Kayıtları cümlelere böl ve yazıya dök (Whisper, 5-10 dk)

In [ ]:
import whisper, subprocess
asr = whisper.load_model("large-v3", device="cuda")

def wav(src, dst, a=None, b=None):
    cmd = ["ffmpeg", "-y", "-v", "error"] + (["-ss", f"{a:.2f}", "-to", f"{b:.2f}"] if a is not None else []) + ["-i", src, "-ac", "1", "-ar", "22050", dst]
    subprocess.run(cmd, check=True)

veri = {}  # dil -> [(dosya_id, metin)]
for k in kayitlar:
    lang = re.search(r"-(tr|en)", os.path.basename(k)).group(1)
    root = f"{WORK}/data-{lang}"; os.makedirs(root + "/wavs", exist_ok=True)
    full = f"{WORK}/{os.path.basename(k)}.wav"; wav(k, full)
    r = asr.transcribe(full, language=lang, word_timestamps=True, condition_on_previous_text=False)
    words = [w for s in r["segments"] for w in s.get("words", [])]
    # cümle sonunda ya da 10 sn dolunca kes; 1,5 sn altını at (XTTS en fazla ~11,6 sn alıyor)
    cur = []
    for w in words:
        cur.append(w)
        dur = cur[-1]["end"] - cur[0]["start"]
        if (w["word"].strip()[-1:] in ".?!" and dur > 1.5) or dur > 10:
            text = "".join(x["word"] for x in cur).strip()
            a, b = max(0, cur[0]["start"] - 0.1), cur[-1]["end"] + 0.15
            if 1.5 <= b - a <= 11.5 and not re.search(r"\d", text):
                i = f"{lang}_{len(veri.get(lang, [])):04d}"
                wav(full, f"{root}/wavs/{i}.wav", a, b)
                veri.setdefault(lang, []).append((i, text))
            cur = []
for lang, rows in veri.items():
    with open(f"{WORK}/data-{lang}/metadata.csv", "w", encoding="utf-8") as f:
        f.writelines(f"{i}|{t}|{t}\n" for i, t in rows)
    toplam = sum(float(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", f"{WORK}/data-{lang}/wavs/{i}.wav"], capture_output=True, text=True).stdout) for i, _ in rows)
    print(lang, len(rows), "parça,", round(toplam/60, 1), "dakika. Örnek:", rows[:3])
del asr; import torch; torch.cuda.empty_cache()

## 4. Hazır XTTS-v2 dosyalarını indir

In [ ]:
from TTS.utils.manage import ModelManager
BASE = f"{WORK}/base"; os.makedirs(BASE, exist_ok=True)
hf = "https://huggingface.co/coqui/XTTS-v2/resolve/main/"
need = [hf + n for n in ["model.pth", "config.json", "vocab.json", "dvae.pth", "mel_stats.pth", "speakers_xtts.pth"] if not os.path.exists(f"{BASE}/{n}")]
if need: ModelManager._download_model_files(need, BASE, progress_bar=True)
print(os.listdir(BASE))

## 5. Eğitim (T4'te ~15-30 dk)

In [ ]:
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.xtts_config import XttsAudioConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig

OUT = f"{WORK}/run"
datasets = [BaseDatasetConfig(formatter="ljspeech", dataset_name=f"arjin-{l}", path=f"{WORK}/data-{l}",
                              meta_file_train="metadata.csv", language=l) for l in veri]
args = GPTArgs(max_conditioning_length=132300, min_conditioning_length=66150, max_wav_length=255995, max_text_length=200,
               mel_norm_file=f"{BASE}/mel_stats.pth", dvae_checkpoint=f"{BASE}/dvae.pth",
               xtts_checkpoint=f"{BASE}/model.pth", tokenizer_file=f"{BASE}/vocab.json",
               gpt_num_audio_tokens=1026, gpt_start_audio_token=1024, gpt_stop_audio_token=1025,
               gpt_use_masking_gt_prompt_approach=True, gpt_use_perceiver_resampler=True)
config = GPTTrainerConfig(
    output_path=OUT, model_args=args, datasets=datasets, run_name="xtts-arjin", project_name="xtts", dashboard_logger="tensorboard",
    audio=XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000),
    epochs=EPOCHS, batch_size=BATCH, eval_batch_size=BATCH, batch_group_size=48, num_loader_workers=2,
    eval_split_max_size=256, eval_split_size=0.05, print_step=25, plot_step=100, log_model_step=1000,
    save_step=100000, save_n_checkpoints=1, save_checkpoints=True, save_best_after=0, print_eval=False,
    optimizer="AdamW", optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=LR, lr_scheduler="MultiStepLR", lr_scheduler_params={"milestones": [50000 * 18], "gamma": 0.5, "last_epoch": -1},
    test_sentences=[])
model = GPTTrainer(config)
train, evals = load_tts_samples(config, eval_split=True, eval_split_max_size=256, eval_split_size=0.05)
print(len(train), "eğitim,", len(evals), "değerlendirme örneği")
Trainer(TrainerArgs(restore_path=None, skip_train_epoch=False, start_with_eval=False, grad_accum_steps=1),
        config, output_path=OUT, model=model, train_samples=train, eval_samples=evals).fit()

## 6. Modeli paketle ve Drive'a kaydet

In [ ]:
import torch
ck = sorted(glob.glob(f"{OUT}/*/best_model*.pth") or glob.glob(f"{OUT}/*/checkpoint_*.pth"), key=os.path.getmtime)[-1]
print("kullanılan:", ck)
sd = torch.load(ck, map_location="cpu", weights_only=False)["model"]
sd = {k: v for k, v in sd.items() if "dvae" not in k and "torch_mel_spectrogram" not in k}  # eğitim artıkları
PKG = f"{WORK}/xtts-arjin"; os.makedirs(PKG + "/refs", exist_ok=True)
torch.save({"model": sd}, f"{PKG}/model.pth")
for n in ["config.json", "vocab.json", "speakers_xtts.pth"]: shutil.copy(f"{BASE}/{n}", PKG)
# en uzun 6 Türkçe parça referans olarak (Mac'te XTTS_SPEAKER_WAV bu klasör olur)
lang0 = "tr" if "tr" in veri else list(veri)[0]
uz = sorted(glob.glob(f"{WORK}/data-{lang0}/wavs/*.wav"), key=os.path.getsize)[-6:]
for f in uz: shutil.copy(f, PKG + "/refs")
shutil.make_archive(f"{WORK}/xtts-arjin", "zip", PKG)
shutil.copy(f"{WORK}/xtts-arjin.zip", DRIVE)
print("Drive'a kaydedildi:", DRIVE + "/xtts-arjin.zip", round(os.path.getsize(f"{WORK}/xtts-arjin.zip")/1e9, 2), "GB")

## 7. Dinleyerek test et

In [ ]:
from TTS.api import TTS
from IPython.display import Audio, display
tts = TTS(model_path=PKG, config_path=PKG + "/config.json").to("cuda")
refs = sorted(glob.glob(PKG + "/refs/*.wav"))
for lang, t in [("tr", "Hiç düşündün mü, bunu ilk kim yaptı? Cevap seni şaşırtacak. Beğendiyseniz beğenip takip etmeyi unutmayın."),
                ("en", "Have you ever wondered who did this first? If you enjoyed this, like and follow for more.")]:
    tts.tts_to_file(text=t, speaker_wav=refs, language=lang, file_path=f"{WORK}/test-{lang}.wav", temperature=0.6)
    display(Audio(f"{WORK}/test-{lang}.wav"))

## 8. Mac'te kullanmak

1. Drive'dan `xtts-arjin.zip` dosyasını indir, `~/xtts-models/arjin` klasörüne aç.
2. Video akışında şu ayarlar kullanılır (bunu ben ayarlarım):
   `XTTS_MODEL_DIR=~/xtts-models/arjin` ve `XTTS_SPEAKER_WAV=~/xtts-models/arjin/refs`